1c. This problem is asking us to set q = 0.4, p = 0. 35, and s = 0.25. Then simulate the game 100,000 times, starting with 10 dollars and calculate the expected value from b, and compare the numerical answer to the theoretical answer. 

In [6]:
import random

def play_one_game(start_i: int, p: float, q: float, s: float) -> int:
    """
    Simulate one run of the Gamblers ruin with retirement. 
    Returns the ending money (0 if ruined, or current i if retired)
    """

    i = start_i
    while True:
        if i == 0:
            return 0
        
        u = random.random()
        if u < s:
            return i
        elif u < s + p:
            i += 1
        else:
            i -= 1

def simulate(n_trials: int = 100000, start_i: int = 10, p: float = 0.35, q: float = 0.4, s: float = 0.25, seed: int = 0):
    random.seed(seed)
    endings = [play_one_game(start_i, p, q, s) for _ in range(n_trials)]
    mean_end = sum(endings) / n_trials

    return mean_end

In [7]:
simulate()

9.80273

Note here we have simulated an Expected ending money of 9.80273. This is extremely close to the theoretical value from part b. If we plug into the theoretical value from part b, using i = 10, p = 0.35, q = 0.4, s = 0.25, we get a value of 9.8. This is extremely close to our simulated value. 

2a. We will compute the transition probbility matrix via python, rather than doing it by hadn. Note howeve the P(i -> J) is (i choose j) * (0.9)^j * (0.1)^i-j.

In [11]:
import math 
import pandas as pd

p_break = 0.1
p_survival = 0.9

states = list(range(6)) # This is 0 ... 5
P = [[0.0 for _ in states] for __ in states]

for i in states:
    if i == 0:
        P[i][5] = 1.0
    else:
            for j in range(i + 1):
                P[i][j] = math.comb(i, j) * (p_survival ** j) * (p_break ** (i-j))

df = pd.DataFrame(P, index = states, columns = states)
df.index.name = "from i working"
df.columns.name = "to j working"

df.round(4)



to j working,0,1,2,3,4,5
from i working,,,,,,
0,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
1,0.1000,0.9000,0.0000,0.0000,0.0000,0.0000
2,0.0100,0.1800,0.8100,0.0000,0.0000,0.0000
3,0.0010,0.0270,0.2430,0.7290,0.0000,0.0000
4,0.0001,0.0036,0.0486,0.2916,0.6561,0.0000
5,0.0000,0.0005,0.0081,0.0729,0.3280,0.5905


2b. Now we must find the expected time until the newly installed machines fail.

In [16]:
import math
import numpy as np
p_break = 0.1
p_survive = 0.9

P = np.zeros((6, 6))
for i in range(6):
    if i == 0:
        P[i, 0] = 1.0  # absorbing at 0 for the hitting-time problem
    else:
        for j in range(i + 1):
            P[i, j] = math.comb(i, j) * (p_survive ** j) * (p_break ** (i - j))

A = np.eye(6)
b = np.zeros(6)

A[0, :] = 0
A[0, 0] = 1
b[0] = 0

for i in range(1, 6):
    A[i, :] -= P[i, :]
    b[i] = 1
h = np.linalg.solve(A,b)
h_5 = h[5]
h_5

np.float64(22.171622659565664)

So we have found the expected number of weeks is around 22.17 or 22 rounded to a whole number. 

2c. Now we are asked to find in the long run the probability that just 1 macine is working. Note that now we must incorporate back in the original chain with the reset

In [ ]:
import math
import numpy as np

p_break = 0.1
p_survive = 0.9

P = np.zeros((6, 6))
for i in range(6):
    if i == 0:
        P[i, 5] = 1.0  
    else:
        for j in range(i + 1):
            P[i, j] = math.comb(i, j) * (p_survive ** j) * (p_break ** (i - j))

# Solve (P^T - I) pi = 0 with sum(pi)=1
A = P.T - np.eye(6)
A[-1, :] = 1.0
b = np.zeros(6)
b[-1] = 1.0

pi = np.linalg.solve(A, b)
pi, pi[1]

(array([0.04315624, 0.40960542, 0.20480677, 0.13651855, 0.10052797,
        0.10538506]),
 np.float64(0.40960542463739175))

So the probability in the longrun that only one machine is working, is 40.96% or 0.4096. 